# Classification: Trial-Level Bias Score (TL-BS) Features

Extends the Zvielli et al. (2015) TL-BS framework to the three-pair
design used in this thesis. The canonical single-pair Zvielli specification is
computed on the depression-canonical `neg_vs_pos` trials, and separately each of
the three pair types provides an independent bias stream for the extended analysis.

Feature sets compared:

1. **TL-BS Single Pair (neg_vs_pos)** — 7 parameters, canonical Zvielli replication
   on the negative-vs-positive stream only.
2. **TL-BS Three-Pair** — 7 parameters × 3 pair types = 21 features.
3. **Traditional Static** — 57 session-level aggregates from `build_static_aggregation`,
   matching notebook 2. Serves as the non-temporal baseline.
4. **Combined** — Traditional Static + Three-Pair TL-BS = 78 features.

## Limitations

Webcam sampling rate (30-60 Hz) limits the temporal resolution of single-trial bias
estimates. Per-pair trial streams (~5-8 trials per session per pair) are shorter than
the ~100-trial dot-probe streams of the original Zvielli work, so individual TL-BS
parameters `peak_pos`, `peak_neg`, and `slope` have higher sampling
noise than in lab-grade eye-tracking studies.

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

from src.evaluation.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance, build_comparison_table,
)
from src.features.session_aggregation import build_static_aggregation
from src.features.tlbs import compute_tlbs_per_pair, tlbs_feature_names, TLBS_PARAMS
from src.visualization.io import save_figure

## 1. Build session-level dataset

### 1.1 Load and filter scene metrics

In [0]:
VALID_PAIRS = ["negative_vs_positive", "negative_vs_neutral", "neutral_vs_positive"]
DURATION_MIN_MS = 500
DURATION_MAX_MS = 10000

scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = (scene_metrics
    .filter(F.col("scene_type") == "stimulus")
    .filter(F.col("scene_quality_valid") == True)
    .filter(F.col("scene_valence_pair").isin(VALID_PAIRS))
    .filter(F.col("duration_ms").between(DURATION_MIN_MS, DURATION_MAX_MS)))

print(f"Clean stimulus scenes: {df_stimulus.count()}")

### 1.2 Traditional static features

In [0]:
agg = build_static_aggregation()
df_traditional = (df_stimulus.groupBy("session_id").agg(*agg.exprs)).toPandas()
print(f"Traditional static features: {len(agg.all_columns)} across {len(df_traditional)} sessions")

### 1.3 Pull trial-level bias values

In [0]:
df_trials = df_stimulus.select("session_id", "scene_index", "scene_valence_pair","bias_neg_vs_pos", "bias_neg_vs_neu", "bias_pos_vs_neu").toPandas()

print(f"Trial-level rows: {len(df_trials)}, sessions: {df_trials['session_id'].nunique()}")

### 1.4 Compute TL-BS parameters per session per pair

In [0]:
df_tlbs = compute_tlbs_per_pair(df_trials)
print(f"TL-BS features computed for {len(df_tlbs)} sessions")
print(f"Parameters per pair: {len(TLBS_PARAMS)} ({TLBS_PARAMS})")
print(f"Total TL-BS features: {df_tlbs.shape[1] - 1}")

### 1.5 Visualize trial-level bias signal (neg_vs_pos only)

In [0]:
df_forms_viz = df_forms.select("session_id", "phq9_score").toPandas()

def _viz_severity(s):
    if s <= 4:  return "minimal"
    if s <= 14: return "moderate"
    return "severe"

df_forms_viz["viz_severity"] = df_forms_viz["phq9_score"].apply(_viz_severity)
df_viz = df_trials[df_trials["scene_valence_pair"] == "negative_vs_positive"].merge(
    df_forms_viz, on="session_id"
)

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)

for ax, sev in zip(axes, ["minimal", "moderate", "severe"]):
    subset = df_viz[df_viz["viz_severity"] == sev]
    sample_sessions = subset["session_id"].unique()[:5]
    for sid in sample_sessions:
        sess = subset[subset["session_id"] == sid].sort_values("scene_index")
        vals = sess["bias_neg_vs_pos"].values
        ax.plot(range(len(vals)), vals, alpha=0.5, linewidth=1.5)
    ax.axhline(y=0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"PHQ-9: {sev}")
    ax.set_xlabel("Trial number (neg_vs_pos only)")
    ax.set_ylabel("bias_neg_vs_pos" if ax == axes[0] else "")

plt.suptitle("Trial-level bias signal on neg_vs_pos scenes", fontsize=13)
plt.tight_layout()
save_figure(fig, name="phq9_tlbs_signal_by_severity", subfolder="tlbs")
plt.show()

### 1.6 Merge and aggregate to user level

In [0]:
df = df_tlbs.merge(df_traditional, on="session_id")
df = df.merge(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score").toPandas(),
    on="session_id",
)

TLBS_ALL = tlbs_feature_names()
TLBS_SINGLE = tlbs_feature_names(["neg_vs_pos"])
TRADITIONAL_FEATURES = agg.all_columns

FEATURE_COLS = TLBS_ALL + TRADITIONAL_FEATURES
NUMERIC_COLS = FEATURE_COLS + ["phq9_score", "bdi_score"]
df = df.groupby("uid", as_index=False)[NUMERIC_COLS].mean()

print(f"After per-user aggregation: {len(df)} users")
print(f"PHQ-9: min={df['phq9_score'].min():.1f}, max={df['phq9_score'].max():.1f}, mean={df['phq9_score'].mean():.1f}")
print(f"BDI-II: min={df['bdi_score'].min():.1f}, max={df['bdi_score'].max():.1f}, mean={df['bdi_score'].mean():.1f}")

## 2. Define targets and feature sets

### 2.1 PHQ-9 targets

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)

def phq9_severity_from_score(s):
    if s <= 4:  return 0
    if s <= 9:  return 1
    if s <= 14: return 2
    if s <= 19: return 3
    return 4

df["phq9_severity_class"] = df["phq9_score"].apply(phq9_severity_from_score)

print("PHQ-9")
print(f"Binary (>=10): {df['phq9_depressed'].value_counts().sort_index().to_dict()}")
print(f"Multi-class: {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

### 2.2 BDI-II targets

In [0]:
df["bdi_depressed"] = (df["bdi_score"] >= 14).astype(int)

def bdi_severity_from_score(s):
    if s <= 13: return 0
    if s <= 19: return 1
    if s <= 28: return 2
    return 3

df["bdi_severity_class"] = df["bdi_score"].apply(bdi_severity_from_score)

print("BDI-II")
print(f"Binary (>=14): {df['bdi_depressed'].value_counts().sort_index().to_dict()}")
print(f"Multi-class: {df['bdi_severity_class'].value_counts().sort_index().to_dict()}")

### 2.3 Feature sets

In [0]:
FEATURE_SETS = {
    "TL-BS single pair (neg_vs_pos)": TLBS_SINGLE,
    "TL-BS three-pair": TLBS_ALL,
    "Traditional static": TRADITIONAL_FEATURES,
    "Combined": TRADITIONAL_FEATURES + TLBS_ALL,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
target_cols = [
    "phq9_score", "phq9_depressed", "phq9_severity_class",
    "bdi_score", "bdi_depressed", "bdi_severity_class",
]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. TL-BS univariate correlations

Reference correlations for `avg_bias_*` are from notebook 2 and serve as the static-feature benchmarks that TL-BS must beat to demonstrate incremental value.

### 4.1 PHQ-9

In [0]:
print("TL-BS parameter correlations with PHQ-9:\n")
for feat in TLBS_ALL:
    r, p = scipy_stats.spearmanr(df_clean[feat], df_clean["phq9_score"])
    star = "*" if p < 0.05 else " "
    print(f"  {star} r={r:+.3f}  p={p:.2e}  {feat}")

print()
for pair_suffix in ["neg_vs_pos", "neg_vs_neu", "pos_vs_neu"]:
    col = f"avg_bias_{pair_suffix}"
    r, p = scipy_stats.spearmanr(df_clean[col], df_clean["phq9_score"])
    print(f"Reference: {col:<25} r={r:+.3f}  p={p:.2e}")

### 4.2 BDI-II

In [0]:
print("TL-BS parameter correlations with BDI-II:\n")
for feat in TLBS_ALL:
    r, p = scipy_stats.spearmanr(df_clean[feat], df_clean["bdi_score"])
    star = "*" if p < 0.05 else " "
    print(f"  {star} r={r:+.3f}  p={p:.2e}  {feat}")

print()
for pair_suffix in ["neg_vs_pos", "neg_vs_neu", "pos_vs_neu"]:
    col = f"avg_bias_{pair_suffix}"
    r, p = scipy_stats.spearmanr(df_clean[col], df_clean["bdi_score"])
    print(f"Reference: {col:<25}  r={r:+.3f}  p={p:.2e}")

## 5. Binary classification

### 5.1 PHQ-9 cutoff (≥ 10)

#### 5.1.1 Run classification

In [0]:
y_phq9_binary = df_clean["phq9_depressed"].values
phq9_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups)

#### 5.1.2 Results

In [0]:
print(phq9_binary_df.to_string(index=False))

#### 5.1.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups, phq9_binary_df, save_name="phq9_clinical_tlbs_best")

### 5.2 PHQ-9 extremes (minimal vs severe)

#### 5.2.1 Run classification

In [0]:
df_phq9_extreme = df_clean[(df_clean["phq9_score"] <= 4) | (df_clean["phq9_score"] >= 20)].copy()
df_phq9_extreme["phq9_extreme"] = (df_phq9_extreme["phq9_score"] >= 20).astype(int)

groups_phq9_extreme = df_phq9_extreme["uid"].values

print(f"PHQ-9 extremes dataset: {len(df_phq9_extreme)} users")
print(f"  Minimal (<=4):  {(df_phq9_extreme['phq9_extreme'] == 0).sum()}")
print(f"  Severe  (>=20): {(df_phq9_extreme['phq9_extreme'] == 1).sum()}")

y_phq9_extreme = df_phq9_extreme["phq9_extreme"].values
phq9_extreme_df = run_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme)

#### 5.2.2 Results

In [0]:
print(phq9_extreme_df.to_string(index=False))

#### 5.2.3 Best model

In [0]:
plot_best_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme, phq9_extreme_df, save_name="phq9_extreme_tlbs_best")

### 5.3 BDI-II cutoff (≥ 14)

#### 5.3.1 Run classification

In [0]:
y_bdi_binary = df_clean["bdi_depressed"].values
bdi_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups)

#### 5.3.2 Results

In [0]:
print(bdi_binary_df.to_string(index=False))

#### 5.3.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups, bdi_binary_df, save_name="bdi_clinical_tlbs_best")

### 5.4 BDI-II extremes

#### 5.4.1 Run classification

In [0]:
df_bdi_extreme = df_clean[(df_clean["bdi_score"] <= 13) | (df_clean["bdi_score"] >= 29)].copy()
df_bdi_extreme["bdi_extreme"] = (df_bdi_extreme["bdi_score"] >= 29).astype(int)

groups_bdi_extreme = df_bdi_extreme["uid"].values

print(f"BDI-II extremes dataset: {len(df_bdi_extreme)} users")
print(f"Minimal (<=13): {(df_bdi_extreme['bdi_extreme'] == 0).sum()}")
print(f"Severe  (>=29): {(df_bdi_extreme['bdi_extreme'] == 1).sum()}")
print()

y_bdi_extreme = df_bdi_extreme["bdi_extreme"].values
bdi_extreme_df = run_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme)

#### 5.4.2 Results

In [0]:
print(bdi_extreme_df.to_string(index=False))

#### 5.4.3 Best model

In [0]:
plot_best_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme, bdi_extreme_df, save_name="bdi_extreme_tlbs_best")

## 6. Multi-class classification

### 6.1 PHQ-9 (5 severity classes)

#### 6.1.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_phq9_multi = df_clean["phq9_severity_class"].values.astype(int)
phq9_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups)

#### 6.1.2 Results

In [0]:
print(phq9_multi_df.to_string(index=False))

#### 6.1.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups, phq9_multi_df, PHQ9_LABELS, save_name="phq9_multiclass_tlbs_best")

### 6.2 BDI-II (4 severity classes)

#### 6.2.1 Run classification

In [0]:
BDI_LABELS = ["Minimal", "Mild", "Moderate", "Severe"]
y_bdi_multi = df_clean["bdi_severity_class"].values.astype(int)
bdi_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups)

#### 6.2.2 Results

In [0]:
print(bdi_multi_df.to_string(index=False))

#### 6.2.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups, bdi_multi_df, BDI_LABELS, save_name="bdi_multiclass_tlbs_best")

## 7. Regression

### 7.1 PHQ-9 score prediction

#### 7.1.1 Run regression

In [0]:
y_phq9_reg = df_clean["phq9_score"].values
phq9_reg_df = run_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups)

#### 7.1.2 Results

In [0]:
print(phq9_reg_df.to_string(index=False))

#### 7.1.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups, phq9_reg_df, score_name="PHQ-9", save_name="phq9_regression_tlbs_best")

### 7.2 BDI-II score prediction

#### 7.2.1 Run regression

In [0]:
y_bdi_reg = df_clean["bdi_score"].values
bdi_reg_df = run_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups)

#### 7.2.2 Results

In [0]:
print(bdi_reg_df.to_string(index=False))

#### 7.2.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups, bdi_reg_df, score_name="BDI-II", save_name="bdi_regression_tlbs_best")

## 8. PHQ-9 vs BDI-II convergent-validity comparison

In [0]:
phq9_results = {
    "Binary cutoff":   (phq9_binary_df,  "auc_roc"),
    "Binary extremes": (phq9_extreme_df, "auc_roc"),
    "Multi-class":     (phq9_multi_df,   "f1_weighted"),
    "Regression":      (phq9_reg_df,     "r2"),
}
bdi_results = {
    "Binary cutoff":   (bdi_binary_df,   "auc_roc"),
    "Binary extremes": (bdi_extreme_df,  "auc_roc"),
    "Multi-class":     (bdi_multi_df,    "f1_weighted"),
    "Regression":      (bdi_reg_df,      "r2"),
}

comparison_df = build_comparison_table(phq9_results, bdi_results)
print(comparison_df.to_string(index=False))

## 9. Feature importance

In [0]:
plot_feature_importance(df_clean, FEATURE_SETS["Combined"], y_phq9_binary, title="Feature importance (PHQ-9 binary, combined)", save_name="phq9_tlbs_combined_feature_importance_top20")

## 10. Summary

### 10.1 PHQ-9

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(phq9_binary_df, phq9_multi_df, phq9_reg_df, feature_order, title="PHQ-9 — TL-BS features", save_name="phq9_tlbs_summary_3panel")

### 10.2 BDI-II

In [0]:
plot_summary(bdi_binary_df, bdi_multi_df, bdi_reg_df, feature_order, title="BDI-II — TL-BS features", save_name="bdi_tlbs_summary_3panel")